# Tokamak Disruption Simulator -- Colab Pipeline

**Purpose:** Generate synthetic tokamak disruption training data using
FreeGSNKE (MHD equilibrium) + TORAX (1D transport) coupled simulation.

**Output:** HDF5 dataset saved to Google Drive with:
- Normal shots (label=0): Te, ne, j, q profiles over time
- Disruptive shots (label=1): pre-disruption evolution + trigger crossing

## Steps
1. Install dependencies (FreeGSNKE, TORAX, etc.)
2. Verify imports and JAX availability
3. Clone the repository and DINA-IMAS machine data
4. Build the ITER machine and compute initial equilibrium
5. **TORAX Validation Tests:**
   - Test A: TORAX standalone (built-in ITER example)
   - Test B: TORAX with our FreeGSNKE GEQDSK geometry
   - Test C: TORAX trajectory with time-dependent geometry
6. Step-by-step pipeline test (simplified transport for trigger detection)
7. Batch dataset generation
8. Save to Google Drive
9. Dataset validation

## Step 1: Install Dependencies

In [ ]:
# ============================================================
# INSTALL CELL — installs packages if missing.
# Safe to re-run: skips if already installed.
# ============================================================

import importlib.util, subprocess, re, os, warnings
warnings.filterwarnings("ignore", message="JAX plugin")

def _pip(*args):
    subprocess.run(["pip", "install", "-q"] + list(args), check=False)

need_torax = importlib.util.find_spec("torax") is None
need_freegs = importlib.util.find_spec("freegsnke") is None

if need_torax or need_freegs:
    print("Installing packages...")

    if need_torax:
        print("  [1/5] Installing TORAX + JAX...")
        _pip("torax")

    # Install matching JAX CUDA plugin (torax pulls jax 0.9.x but
    # Colab's pre-installed cuda plugin is 0.7.x — incompatible)
    print("  [2/5] Installing JAX CUDA support...")
    _pip("jax[cuda12]")

    if need_freegs:
        print("  [3/5] Installing FreeGSNKE (bypassing numpy pin)...")
        _pip("freegs4e", "--no-deps")
        _pip("freegsnke", "--no-deps")

    # Runtime deps that --no-deps skipped + shared deps
    print("  [4/5] Installing scipy / matplotlib / h5py / xarray / deepdiff...")
    _pip("scipy", "matplotlib", "h5py", "netCDF4", "xarray", "deepdiff")

    # Force-reinstall numpy so .py AND .so C-extensions match.
    r = subprocess.run(["pip", "show", "numpy"], capture_output=True, text=True)
    m = re.search(r"Version: ([\d.]+)", r.stdout)
    nv = m.group(1) if m else "2.4.3"
    print(f"  [5/5] Force-reinstalling numpy {nv} to fix ABI...")
    _pip(f"numpy=={nv}", "--force-reinstall", "--no-deps")

    # Uninstall numba — incompatible with numpy>=2.0 and causes freegs4e crash
    subprocess.run(["pip", "uninstall", "numba", "-y", "-q"], check=False)

    print("\nDone. Restarting kernel — re-run all cells after reconnect.")
    os.kill(os.getpid(), 9)

else:
    import numpy as np, jax
    print(f"Packages already installed.")
    print(f"  numpy  : {np.__version__}")
    print(f"  jax    : {jax.__version__}")

## Step 2: Verify Imports

In [ ]:
import numpy as np

# Numpy 2.x compatibility patch for freegs4e/freegsnke.
# These packages were written for numpy 1.x and use type aliases
# (np.float, np.int, np.bool, np.complex) that were removed in numpy 2.0.
for _name, _builtin in [('float', float), ('int', int),
                         ('bool', bool), ('complex', complex),
                         ('object', object), ('str', str)]:
    if not hasattr(np, _name):
        setattr(np, _name, _builtin)

# Fake numba module — Colab ships numba which is incompatible with numpy>=2.0.
# freegs4e.critical tries `from numba import njit` and crashes in the except
# handler (missing `import warnings`). Injecting a fake numba avoids both issues.
import types, sys
_fake_numba = types.ModuleType("numba")
def _njit(*args, **kwargs):
    if args and callable(args[0]):
        return args[0]
    def _decorator(func):
        return func
    return _decorator
_fake_numba.njit = _njit
sys.modules["numba"] = _fake_numba

# Now import everything
import freegsnke
import freegs4e
import jax
import torax
import h5py

print(f"numpy     : {np.__version__}")
print(f"freegsnke : {freegsnke.__version__}")
print(f"freegs4e  : {freegs4e.__version__}")
print(f"jax       : {jax.__version__}")
print(f"torax     : {torax.__version__}")
print()

# Quick JAX smoke test
x = jax.numpy.array([1.0, 2.0, 3.0])
print(f"JAX OK -- devices: {jax.devices()}")

## Step 3: Clone Repository & Set Up Paths

In [ ]:
import os

REPO_URL  = "https://github.com/BecerraMiguel/tokamak-disruption-simulator.git"
REPO_DIR  = "/content/tokamak-disruption-simulator"
DINA_URL  = "https://github.com/iterorganization/DINA-IMAS.git"
DINA_DIR  = "/content/DINA-IMAS"

# Clone the simulator repo
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Clone DINA-IMAS for the ITER machine config
if not os.path.exists(DINA_DIR):
    # Only need the machines/ directory -- use sparse checkout for speed
    !git clone --no-checkout --depth=1 {DINA_URL} {DINA_DIR}
    !cd {DINA_DIR} && git sparse-checkout init --cone
    !cd {DINA_DIR} && git sparse-checkout set machines
    !cd {DINA_DIR} && git checkout

# Add repo src/ to Python path
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

print("Repository ready.")
print(f"ITER config: {DINA_DIR}/machines/iter/tokamak_config.dat")
assert os.path.exists(f"{DINA_DIR}/machines/iter/tokamak_config.dat"), \
    "ITER config not found -- check DINA-IMAS clone"

## Step 4: Build the ITER Machine

In [ ]:
from predisruption.iter_machine import build_iter_machine, ITER_PARAMS

CONFIG_PATH = f"{DINA_DIR}/machines/iter/tokamak_config.dat"

tokamak, active_coils, domain = build_iter_machine(
    config_path=CONFIG_PATH,
    include_passives=False,   # Phase 1: no eddy currents
    verbose=True,
)
print(f"\nDomain: R=[{domain[0]}, {domain[1]}] m, Z=[{domain[2]}, {domain[3]}] m")
print(f"Active circuits: {list(active_coils.keys())}")

## Step 5: Compute Initial ITER Equilibrium

In [ ]:
from predisruption.equilibrium import EquilibriumSolver

eq_solver = EquilibriumSolver(
    tokamak=tokamak,
    nx=65, ny=65,
    Rmin=domain[0], Rmax=domain[1],
    Zmin=domain[2], Zmax=domain[3],
    verbose=True,
)

print("Solving initial 15 MA equilibrium...")
eq = eq_solver.solve_static(
    Ip=15.0e6,
    betap=0.5,
    xpoints=[(6.0, -3.8)],
)

signals = eq_solver.get_signals(eq)
print(f"\nInitial equilibrium:")
print(f"  Ip    = {signals['Ip']*1e-6:.2f} MA")
print(f"  q95   = {signals['q95']:.2f}")
print(f"  betaN = {signals['betaN']:.3f}")
print(f"  kappa = {signals['kappa']:.2f}")
print(f"  delta = {signals['delta']:.2f}")

In [ ]:
import matplotlib.pyplot as plt
from predisruption.shot_runner import TRIGGERS

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Poloidal flux map
R = eq.R
Z = eq.Z
psi = eq.psi()
axes[0].contourf(R, Z, psi.T, levels=20, cmap='RdBu_r')
R_sep, Z_sep = signals['R_sep'], signals['Z_sep']
if len(R_sep) > 0:
    axes[0].plot(R_sep, Z_sep, 'k-', lw=2, label='Separatrix')
axes[0].plot(signals['R_axis'], signals['Z_axis'], 'k+', ms=10, label='O-point')
axes[0].set_xlabel('R (m)')
axes[0].set_ylabel('Z (m)')
axes[0].set_title('Poloidal flux psi(R,Z)')
axes[0].set_aspect('equal')
axes[0].legend()

# q profile
rho = signals['rho']
q   = signals['q_profile']
axes[1].plot(rho, q, 'b-', lw=2)
axes[1].axhline(y=1.0, ls='--', c='gray', label='q=1')
axes[1].axhline(y=2.0, ls='--', c='orange', label='q=2')
axes[1].axhline(y=TRIGGERS['q95'], ls='--', c='red', label=f"q95 limit={TRIGGERS['q95']}")
axes[1].set_xlabel('rho = sqrt(Phi/Phi_edge)')
axes[1].set_ylabel('q(rho)')
axes[1].set_title('Safety factor profile')
axes[1].legend()
axes[1].set_ylim(0, 8)

plt.tight_layout()
os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
plt.savefig(f"{REPO_DIR}/results/iter_initial_eq.png", dpi=120)
plt.show()
print("Saved to results/iter_initial_eq.png")

---

# TORAX Validation Tests

Three progressive tests to validate that TORAX works on Colab with our geometry:

- **Test A**: TORAX standalone using a built-in ITER example (no custom geometry)
- **Test B**: TORAX with our FreeGSNKE GEQDSK geometry (single equilibrium)
- **Test C**: TORAX with time-dependent geometry from the coupled FreeGSNKE simulator

Each test builds on the previous one. If Test A fails, the issue is with TORAX/JAX itself.
If Test B fails, the issue is with our GEQDSK file format or TORAX config.
If Test C fails, the issue is with the coupling workflow.

## Test A: TORAX Standalone (Built-in ITER Example)

Run TORAX with its built-in ITER hybrid ramp-up configuration (no custom geometry).
This verifies that TORAX + JAX work correctly on this Colab instance.

In [ ]:
import torax
from torax.examples.iterhybrid_rampup import CONFIG as ITER_HYBRID_CONFIG
import copy

# Run with shorter time for quick test
config = copy.deepcopy(ITER_HYBRID_CONFIG)
config['numerics']['t_final'] = 10  # 10 seconds only
torax_config = torax.ToraxConfig.from_dict(config)
print("Running TORAX standalone test (ITER hybrid, 10s)...")
data_tree, state_history = torax.run_simulation(torax_config)
print(f"TORAX standalone OK: {len(data_tree['profiles'].ds.coords['time'])} time steps")

# Plot Te and ne profiles at final time
profiles = data_tree["profiles"].ds
Te_final = profiles["T_e"].values[-1, :]
ne_final = profiles["n_e"].values[-1, :]

# Use the matching rho grid for each variable (cell vs face can differ in size)
Te_rho = profiles["T_e"].dims[-1]  # last dim name, e.g. 'rho_cell_norm' or 'rho_face_norm'
ne_rho = profiles["n_e"].dims[-1]
rho_Te = profiles.coords[Te_rho].values
rho_ne = profiles.coords[ne_rho].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rho_Te, Te_final, 'r-', lw=2)
axes[0].set(xlabel='rho', ylabel='Te (keV)', title='TORAX standalone: Te profile')
axes[0].grid(True, alpha=0.3)
axes[1].plot(rho_ne, ne_final * 1e-20, 'b-', lw=2)
axes[1].set(xlabel='rho', ylabel='ne (10^20 m^-3)', title='TORAX standalone: ne profile')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Test A PASSED")

## Test B: TORAX with FreeGSNKE GEQDSK Geometry

Write the FreeGSNKE equilibrium to a GEQDSK file, then run TORAX using that
geometry instead of its built-in circular or CHEASE geometry. Compare initial
profiles with the simplified transport model.

In [ ]:
import os
import tempfile

# Write our FreeGSNKE equilibrium to GEQDSK
geqdsk_dir = tempfile.mkdtemp(prefix="torax_test_")
geqdsk_file = "iter_test.geqdsk"
eq_solver.write_geqdsk(os.path.join(geqdsk_dir, geqdsk_file), eq)
print(f"GEQDSK written to {geqdsk_dir}/{geqdsk_file}")

# Build TORAX config with our GEQDSK geometry
from predisruption.transport import _default_iter_torax_config

config_eqdsk = _default_iter_torax_config(
    geometry_dir=geqdsk_dir,
    geometry_files={0.0: geqdsk_file},
    t_final=5.0,         # short test
    Ip_A=15.0e6,
    T_e0_keV=10.0,
    P_heat_W=33e6,
    transport_model="constant",  # start simple
)

print("Running TORAX with FreeGSNKE GEQDSK (constant transport, 5s)...")
torax_config = torax.ToraxConfig.from_dict(config_eqdsk)
data_tree_eqdsk, _ = torax.run_simulation(torax_config)
print(f"TORAX GEQDSK test OK: {len(data_tree_eqdsk['profiles'].ds.coords['time'])} time steps")

# Compare with simplified transport
from predisruption.transport import SimplifiedTransport
simple = SimplifiedTransport(n_rho=51)
simple_state = simple.init(Ip_A=15e6, T_e0_keV=10.0, n_e0_m3=1e20, geometry=signals)

profiles_eqdsk = data_tree_eqdsk["profiles"].ds
rho_torax = profiles_eqdsk.coords["rho_cell_norm"].values
rho_simple = simple_state.rho

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# Te comparison
axes[0].plot(rho_torax, profiles_eqdsk["T_e"].values[0, :], 'r-', lw=2, label='TORAX (t=0)')
axes[0].plot(rho_simple, simple_state.T_e, 'b--', lw=2, label='Simplified')
axes[0].set(xlabel='rho', ylabel='Te (keV)', title='Te comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ne comparison
axes[1].plot(rho_torax, profiles_eqdsk["n_e"].values[0, :] * 1e-20, 'r-', lw=2, label='TORAX')
axes[1].plot(rho_simple, simple_state.n_e * 1e-20, 'b--', lw=2, label='Simplified')
axes[1].set(xlabel='rho', ylabel='ne (10^20 m^-3)', title='ne comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# q comparison
if "q" in profiles_eqdsk:
    rho_face = profiles_eqdsk.coords["rho_face_norm"].values
    axes[2].plot(rho_face, profiles_eqdsk["q"].values[0, :], 'r-', lw=2, label='TORAX')
axes[2].plot(rho_simple, simple_state.q, 'b--', lw=2, label='Simplified/FreeGSNKE')
axes[2].set(xlabel='rho', ylabel='q', title='Safety factor comparison')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Test B: TORAX vs Simplified Transport (initial profiles)', fontsize=13)
plt.tight_layout()
plt.show()
print("Test B PASSED")

## Test C: TORAX Trajectory with Time-Dependent Geometry

Use the full coupling workflow: pre-compute FreeGSNKE equilibria at multiple
time points, write GEQDSK files, then run TORAX in full-trajectory mode with
time-dependent geometry. This is the production workflow for high-fidelity data.

In [ ]:
from predisruption.transport import TransportSolver
from predisruption.coupling import CoupledSimulator

# Create TORAX-backed transport solver
tr_solver_torax = TransportSolver(backend="torax", n_rho=51)

# Create coupled simulator
simulator = CoupledSimulator(
    eq_solver=eq_solver,
    tr_solver=tr_solver_torax,
    dt_couple=1.0,
    verbose=True,
)

# Define waveforms
def Ip_wave(t): return 15e6  # flat-top
def P_wave(t): return 33e6   # constant heating
def ne_wave(t): return 1e20  # constant density target

print("Running TORAX trajectory with pre-computed FreeGSNKE geometry...")
trajectory = simulator.run_with_torax(
    t_end=30.0,
    Ip_waveform=Ip_wave,
    P_heat_waveform=P_wave,
    n_target_waveform=ne_wave,
    n_eq_steps=6,            # 6 equilibria over 30s
    transport_model="constant",
)
simulator.cleanup()

print(f"\nTrajectory complete: {len(trajectory['time'])} time steps")
print(f"  Te(0) range: {trajectory['T_e'][0,:].min():.1f} - {trajectory['T_e'][0,:].max():.1f} keV")
print(f"  ne(0) range: {trajectory['n_e'][0,:].min():.2e} - {trajectory['n_e'][0,:].max():.2e} m^-3")

# Plot TORAX trajectory
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
t = trajectory['time']

axes[0,0].plot(t, trajectory['Ip']*1e-6, 'b-', lw=2)
axes[0,0].set(ylabel='Ip (MA)', title='Plasma current')

axes[0,1].plot(t, trajectory['q95'], 'g-', lw=2)
axes[0,1].axhline(2.2, ls='--', c='red', label='limit')
axes[0,1].set(ylabel='q95', title='Safety factor at 95%')
axes[0,1].legend()

axes[0,2].plot(t, trajectory['betaN'], 'm-', lw=2)
axes[0,2].axhline(3.2, ls='--', c='red', label='limit')
axes[0,2].set(ylabel='betaN', title='Normalised beta')
axes[0,2].legend()

axes[1,0].plot(t, trajectory['f_GW'], 'c-', lw=2)
axes[1,0].axhline(0.95, ls='--', c='red', label='limit')
axes[1,0].set(ylabel='f_GW', title='Greenwald fraction')
axes[1,0].legend()

# Te profile at last time step
rho_traj = trajectory['rho']
axes[1,1].plot(rho_traj, trajectory['T_e'][:, -1], 'r-', lw=2)
axes[1,1].set(xlabel='rho', ylabel='Te (keV)', title=f'Te profile at t={t[-1]:.0f}s')

axes[1,2].plot(rho_traj, trajectory['n_e'][:, -1]*1e-20, 'b-', lw=2)
axes[1,2].set(xlabel='rho', ylabel='ne (10^20 m^-3)', title=f'ne profile at t={t[-1]:.0f}s')

for ax in axes.flat:
    ax.set_xlabel(ax.get_xlabel() or 't (s)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Test C: TORAX trajectory with FreeGSNKE geometry', fontsize=14)
plt.tight_layout()
os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
plt.savefig(f"{REPO_DIR}/results/torax_trajectory.png", dpi=120)
plt.show()
print("Test C PASSED")

---

## Step-by-Step Pipeline Test (Simplified Transport)

For step-by-step coupling (trigger detection, disruption scenarios), we use the
simplified transport backend. This runs locally without JAX and supports the
`step()` interface needed for trigger detection at each time step.

TORAX runs in full-trajectory mode (see Test C above) for high-fidelity profiles,
but cannot do step-by-step execution.

In [ ]:
from predisruption.transport import TransportSolver
from predisruption.shot_runner import ShotRunner, reference_15MA_scenario, TRIGGERS

# Transport solver -- use simplified backend for step-by-step coupling
tr_solver_simple = TransportSolver(backend="simplified", n_rho=51)

# Shot runner
runner = ShotRunner(
    eq_solver=eq_solver,
    tr_solver=tr_solver_simple,
    verbose=True,
)

# Test shot (30 s flat-top) to verify the pipeline
scenario = reference_15MA_scenario(t_end=30.0, dt=1.0)
result = runner.run_shot(scenario, shot_id=0)

print(f"\nTest shot complete:")
print(f"  label          = {result['label']}")
print(f"  time steps     = {len(result['time'])}")
print(f"  Te(0, t_end)   = {result['T_e'][0, -1]:.1f} keV")
print(f"  ne(0, t_end)   = {result['n_e'][0, -1]:.2e} m^-3")
print(f"  q95(t_end)     = {result['q95'][-1]:.2f}")
print(f"  betaN(t_end)   = {result['betaN'][-1]:.3f}")
print(f"  f_GW(t_end)    = {result['f_GW'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
t = result['time']

axes[0,0].plot(t, result['Ip']*1e-6)
axes[0,0].set(ylabel='Ip (MA)', title='Plasma current')

axes[0,1].plot(t, result['q95'])
axes[0,1].axhline(2.2, ls='--', c='red', label='limit')
axes[0,1].set(ylabel='q95', title='Safety factor at 95%')
axes[0,1].legend()

axes[0,2].plot(t, result['betaN'])
axes[0,2].axhline(3.2, ls='--', c='red', label='limit')
axes[0,2].set(ylabel='betaN', title='Normalised beta')
axes[0,2].legend()

axes[1,0].plot(t, result['f_GW'])
axes[1,0].axhline(0.95, ls='--', c='red', label='limit')
axes[1,0].set(ylabel='f_GW', title='Greenwald fraction')
axes[1,0].legend()

# Te profile at last time step
rho = result['rho']
axes[1,1].plot(rho, result['T_e'][:, -1])
axes[1,1].set(xlabel='rho', ylabel='Te (keV)', title=f'Te profile at t={t[-1]:.0f}s')

# ne profile at last time step
axes[1,2].plot(rho, result['n_e'][:, -1]*1e-20)
axes[1,2].set(xlabel='rho', ylabel='ne (10^20 m^-3)', title=f'ne profile at t={t[-1]:.0f}s')

for ax in axes.flat:
    ax.set_xlabel(ax.get_xlabel() or 't (s)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Test shot -- ITER 15 MA normal (simplified transport)', fontsize=14)
plt.tight_layout()
plt.savefig(f"{REPO_DIR}/results/test_shot.png", dpi=120)
plt.show()

## Batch Dataset Generation

Generate N_normal normal shots + N_disruptive disruptive shots.
Vary: Ip (5-15 MA), P_heat (10-40 MW), ne_target (0.5-1.2 x n_GW).

Uses the simplified transport backend for step-by-step trigger detection.

In [ ]:
# ---- Dataset generation parameters ----
N_NORMAL      = 50    # normal shots per run
N_DISRUPTIVE  = 50    # disruptive shots per run
T_END_NORMAL  = 90.0  # s
T_END_DISRUPT = 75.0  # s
DT_COUPLE     = 1.0   # coupling time step (s)
RNG_SEED      = 42

rng = np.random.default_rng(RNG_SEED)
print(f"Will generate {N_NORMAL} normal + {N_DISRUPTIVE} disruptive shots")

In [ ]:
import h5py
import os
from predisruption.shot_runner import disruptive_scenario, reference_15MA_scenario

OUTPUT_H5 = f"{REPO_DIR}/data/dataset_v1.h5"
os.makedirs(os.path.dirname(OUTPUT_H5), exist_ok=True)

def vary_normal_scenario(rng, t_end, dt):
    """Sample a random normal scenario from the ITER operating space."""
    Ip_flat = rng.uniform(8e6, 15e6)       # 8-15 MA
    P_heat  = rng.uniform(20e6, 40e6)      # 20-40 MW
    ne_frac = rng.uniform(0.5, 0.85)       # 50-85% Greenwald
    a = ITER_PARAMS['a_minor']
    n_GW = (Ip_flat*1e-6) / (np.pi * a**2) * 1e20   # m^-3
    ne_target = ne_frac * n_GW
    cfg = reference_15MA_scenario(t_end=t_end, dt=dt)
    cfg.Ip_val  = np.array([3e6, Ip_flat, Ip_flat, 3e6])
    cfg.P_heat_val = np.array([5e6, P_heat, P_heat])
    cfg.ne_val  = np.array([3e19, ne_target, ne_target])
    return cfg

def vary_disruptive_scenario(rng, t_end, dt):
    """Sample a random disruptive scenario."""
    ptype = rng.choice(["density", "beta", "q95"])
    amp   = rng.uniform(0.2, 0.5)
    t_perturb = rng.uniform(30.0, min(50.0, t_end - 10.0))
    return disruptive_scenario(
        perturbation_type=ptype,
        perturbation_start=t_perturb,
        perturbation_amp=amp,
        t_end=t_end, dt=dt,
    )

def write_shot_to_hdf5(f, result, shot_id):
    """Write one shot result to an open HDF5 file."""
    grp = f.create_group(f"shots/shot_{shot_id:05d}")
    grp.attrs["label"]           = result["label"]
    grp.attrs["disruption_time"] = result["disruption_time"]
    grp.attrs["trigger"]         = result["trigger"]
    grp.attrs["shot_id"]         = shot_id

    grp.create_dataset("time", data=result["time"].astype(np.float32))
    grp.create_dataset("rho",  data=result["rho"].astype(np.float32))

    sig = grp.create_group("signals")
    for key in ["Ip", "betaN", "q95", "f_GW", "W_thermal"]:
        sig.create_dataset(key, data=result[key].astype(np.float32))
    for key in ["T_e", "n_e", "j_tor", "q_profile"]:
        sig.create_dataset(key, data=result[key].astype(np.float32))

# Run generation
with h5py.File(OUTPUT_H5, "w") as hf:
    # Metadata
    meta = hf.create_group("metadata")
    meta.attrs["n_normal"]      = N_NORMAL
    meta.attrs["n_disruptive"]  = N_DISRUPTIVE
    meta.attrs["dt_couple"]     = DT_COUPLE
    meta.attrs["torax_backend"] = str(tr_solver_simple.backend)
    meta.attrs["rng_seed"]      = RNG_SEED

    shot_count = 0

    # --- Normal shots ---
    print(f"Generating {N_NORMAL} normal shots...")
    for i in range(N_NORMAL):
        try:
            scenario = vary_normal_scenario(rng, T_END_NORMAL, DT_COUPLE)
            result   = runner.run_shot(scenario, shot_id=shot_count)
            write_shot_to_hdf5(hf, result, shot_count)
            shot_count += 1
            print(f"  Normal {i+1}/{N_NORMAL} done (shot_{shot_count-1:05d})")
        except Exception as e:
            print(f"  Normal shot {i} FAILED: {e}")

    # --- Disruptive shots ---
    print(f"\nGenerating {N_DISRUPTIVE} disruptive shots...")
    for i in range(N_DISRUPTIVE):
        try:
            scenario = vary_disruptive_scenario(rng, T_END_DISRUPT, DT_COUPLE)
            result   = runner.run_shot(scenario, shot_id=shot_count)
            write_shot_to_hdf5(hf, result, shot_count)
            shot_count += 1
            print(f"  Disruptive {i+1}/{N_DISRUPTIVE} done  "
                  f"trigger={result['trigger']}  "
                  f"t_disrupt={result['disruption_time']:.1f}s")
        except Exception as e:
            print(f"  Disruptive shot {i} FAILED: {e}")

print(f"\nDataset written: {OUTPUT_H5}")
print(f"Total shots: {shot_count}")

## Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DEST = "/content/drive/MyDrive/tokamak_disruption_data/"
os.makedirs(DRIVE_DEST, exist_ok=True)

shutil.copy(OUTPUT_H5, DRIVE_DEST)
print(f"Dataset copied to Google Drive: {DRIVE_DEST}")

## Quick Dataset Validation

In [ ]:
with h5py.File(OUTPUT_H5, "r") as hf:
    shot_keys = list(hf["shots"].keys())
    n_total   = len(shot_keys)

    labels     = [hf[f"shots/{k}"].attrs["label"]    for k in shot_keys]
    triggers   = [hf[f"shots/{k}"].attrs["trigger"]  for k in shot_keys]
    dtimes     = [hf[f"shots/{k}"].attrs["disruption_time"] for k in shot_keys]

    n_normal   = sum(1 for l in labels if l == 0)
    n_disrupt  = sum(1 for l in labels if l == 1)

    from collections import Counter
    trigger_counts = Counter(t for t, l in zip(triggers, labels) if l == 1)

    # Sample shot shape
    sample = hf[f"shots/{shot_keys[0]}"]
    T_e_shape = sample["signals/T_e"].shape
    n_t = len(sample["time"])

print(f"Dataset summary:")
print(f"  Total shots     : {n_total}")
print(f"  Normal  (L=0)   : {n_normal}")
print(f"  Disrupt (L=1)   : {n_disrupt}")
print(f"  Trigger types   : {dict(trigger_counts)}")
print(f"  Te shape [n_rho, n_t] in sample shot: {T_e_shape}")

import numpy as np
valid_dtimes = [d for d, l in zip(dtimes, labels) if l==1 and not np.isnan(d)]
if valid_dtimes:
    print(f"  Disruption time : {np.mean(valid_dtimes):.1f} +/- {np.std(valid_dtimes):.1f} s")